In [0]:
import random
from datetime import date, timedelta

random.seed(42)
CATALOGO = "workspace"

clima_rows = spark.table(f"{CATALOGO}.bronze.clima_raw").collect()
clima = {(r.departamento, r.anio, r.mes): (r.precip_pronosticada_mm, r.precip_real_mm)
         for r in clima_rows}
REF_MM = 130

productos = [("NITRO-XP",40),("NITRO-S-XP",15),("FERTI-CAFE",8),("FERTI-BANANO",7),
             ("FERTI-MAIZ",12),("FERTI-PAPA",6),("FERTI-PASTOS",6),("FERTI-ARROZ",6)]
prod_ids, prod_pesos = [p[0] for p in productos], [p[1] for p in productos]

departamentos = [("Valle",45),("Tolima",15),("Antioquia",15),("Quindio",13),("Narino",12)]
dep_nombres, dep_pesos = [d[0] for d in departamentos], [d[1] for d in departamentos]

clientes = [f"C{str(i).zfill(3)}" for i in range(1,41)]

print("Configuracion lista. Clima cargado:", len(clima), "registros")

In [0]:
fecha_min, fecha_max = date(2023,1,1), date(2025,12,31)
rango_dias = (fecha_max - fecha_min).days

filas = []
for n in range(1, 5001):
    pedido_id    = f"PED-{str(n).zfill(5)}"
    fecha_pedido = fecha_min + timedelta(days=random.randint(0, rango_dias))
    anio, mes    = fecha_pedido.year, fecha_pedido.month
    cliente_id   = random.choice(clientes)
    producto_id  = random.choices(prod_ids, weights=prod_pesos)[0]
    departamento = random.choices(dep_nombres, weights=dep_pesos)[0]

    pron, real  = clima[(departamento, anio, mes)]
    factor_prog = pron / REF_MM
    factor_real = real / REF_MM
    base = random.uniform(80, 100)

    tm_prog = round(base * (factor_prog ** 2) * random.uniform(0.97, 1.03), 1)
    tm_vend = round(base * (factor_real ** 2) * random.uniform(0.97, 1.03), 1)
    tm_fab  = round(tm_vend * random.uniform(0.98, 1.02), 1)
    precio  = round(random.uniform(400, 700), 2)

    repeticiones = random.choices([1,2,3], weights=[80,15,5])[0]
    for r in range(repeticiones):
        fecha_act = fecha_pedido + timedelta(days=r*random.randint(1,5))
        tm_prog_r = round(tm_prog * random.uniform(0.9, 1.1), 1)
    
        dep_final    = None if random.random() < 0.05 else departamento
        precio_final = None if random.random() < 0.05 else precio
        filas.append((pedido_id, fecha_pedido, fecha_act, cliente_id, producto_id,
                      dep_final, tm_prog_r, tm_fab, tm_vend, precio_final))

print("Filas generadas:", len(filas))

bajo = [f[8] for f in filas if f[5] is not None and clima[(f[5], f[1].year, f[1].month)][1] < 100]
alto = [f[8] for f in filas if f[5] is not None and clima[(f[5], f[1].year, f[1].month)][1] > 170]
print("tm_vend lluvia BAJA:", round(sum(bajo)/len(bajo),1))
print("tm_vend lluvia ALTA:", round(sum(alto)/len(alto),1))

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType, DoubleType

schema = StructType([
    StructField("pedido_id", StringType()),
    StructField("fecha_pedido", DateType()),
    StructField("fecha_actualizacion", DateType()),
    StructField("cliente_id", StringType()),
    StructField("producto_id", StringType()),
    StructField("departamento_destino", StringType()),
    StructField("tm_programadas", DoubleType()),
    StructField("tm_fabricadas", DoubleType()),
    StructField("tm_vendidas", DoubleType()),
    StructField("precio_usd_tm", DoubleType()),
])

df_ventas = spark.createDataFrame(filas, schema)

(df_ventas.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOGO}.bronze.ventas_raw"))

print("Tabla bronze.ventas_raw creada")